In [ ]:
# Cellule 1 : Imports et configuration
%load_ext autoreload
%autoreload 2

import tools.ai_token as tk
#import tools.ai_attention_decoder as att
import os
import tools.ai_gpt_ep as gpt
#import random


In [ ]:
# Configuration
base_folder = "data/train/wiki_fr_test"
output_file = "data/corpus_tokenizer_15mb.txt"
num_files_to_merge = 15

# 1. Récupérer tous les fichiers de tous les répertoires
all_files = []

# Parcourir tous les répertoires (AA à CN)
for subdir in os.listdir(base_folder):
    subdir_path = os.path.join(base_folder, subdir)
    
    # Vérifier que c'est bien un répertoire
    if os.path.isdir(subdir_path):
        # Récupérer tous les fichiers .txt du répertoire
        files_in_subdir = [f for f in os.listdir(subdir_path)]
        
        # Ajouter le chemin complet à la liste
        for fname in files_in_subdir:
            full_path = os.path.join(subdir_path, fname)
            all_files.append(full_path)

print(f"Total fichiers trouvés : {len(all_files)}")

# 2. Sélection aléatoire
if len(all_files) > num_files_to_merge:
    selected_files = random.sample(all_files, num_files_to_merge)
else:
    selected_files = all_files
    print(f"Attention : Seulement {len(selected_files)} fichiers disponibles")

print(f"Fusion de {len(selected_files)} fichiers...")

# 3. Fusion
total_size = 0
with open(output_file, 'w', encoding='utf-8') as outfile:
    for file_path in selected_files:
        try:
            with open(file_path, 'r', encoding='utf-8') as infile:
                content = infile.read()
                outfile.write(content + "\n")
                total_size += len(content.encode('utf-8'))
                print(f"✓ Ajouté : {file_path}")
        except Exception as e:
            print(f"✗ Erreur avec {file_path} : {e}")

print(f"\nFichier créé : {output_file}")
print(f"Taille approximative : {total_size / (1024*1024):.2f} MB")

In [ ]:
fileName = 'data/corpus_tokenizer_15mb.txt'
fileName2 = 'data/corpus_francais.txt'

In [ ]:
token = tk.BPETokenizer()
token.load_merges('data/ep_merges_full.json')
print(len(token.vocab))

In [ ]:
tko = tk.OptimizedTokenizer(token.merges)

# Entrainement du model

# Verif

In [ ]:
with open(fileName, 'r', encoding='utf-8') as f:
    text = f.read()

In [ ]:
e = token.encode(text)
print(e[:100])

In [ ]:
e_ = tko.encode(text)
print(e_[:100])

In [ ]:
import glob as glob
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import numpy as np 

In [ ]:
# --- CONFIG ---
data_root = "data/train/wiki_fr_test" # Ton dossier
output_dir = "data/encoded"
os.makedirs(output_dir, exist_ok=True)

# Ton tokenizer (remplace par le tien si c'est une classe custom)
# tokenizer = ... 
# enc = tiktoken.get_encoding("gpt2") 

def process_files_vf():
    # 1. Lister tous les fichiers
    files = glob.glob(os.path.join(data_root, "**", "*"), recursive=True)
    print(f"📚 Trouvé {len(files)} fichiers.")

    # 2. Tout lire dans une liste géante (Attention à la RAM ici si > 10Go de texte)
    # Pour 6000 fichiers wiki, ça tient large sur un M1
    all_tokens = []
    
    print("🔄 Tokenization en cours...")
    for f_path in (files):
        try:
            with open(f_path, 'r', encoding='utf-8') as f:
                text = f.read()
                if len(text) > 0:
                    # Ajout du token de fin de texte (EOT) pour séparer les docs
                    tokens = tko.encode(text) # + [tko.eot_token] 
                    all_tokens.extend(tokens)
        except Exception as e:
            print(f"Skipped {f_path}: {e}")

    total_tokens = len(all_tokens)
    print(f"📊 Total tokens: {total_tokens}")

    # 3. Conversion en numpy array optimisé (uint16 suffit si vocab < 65535)
    # Si ton vocab est > 65535, utilise np.int32
    print("💾 Conversion en binaire...")
    data = np.array(all_tokens, dtype=np.uint16)

    # 4. Split Train (90%) / Val (10%)
    n = int(0.9 * len(data))
    train_data = data[:n]
    val_data = data[n:]

    # 5. Sauvegarde sur disque
    train_data.tofile(os.path.join(output_dir, 'train.bin'))
    val_data.tofile(os.path.join(output_dir, 'val.bin'))
    
    print(f"✅ Terminé ! Fichiers sauvegardés dans {output_dir}")
    print(f"   Train: {len(train_data)/1e6:.2f}M tokens")
    print(f"   Val:   {len(val_data)/1e6:.2f}M tokens")

if __name__ == '__main__':
    process_files_vf()

In [ ]:
# --- EXEMPLE D'UTILISATION ---

# Configuration du modèle (Architecture fixe)
model_config = {
    'n_embd': 384, 'num_heads': 6, 'n_layers': 6, 
    'block_size': 256, 'dropout': 0.2
}

# Paramètres d'entraînement (Modifiables à chaque session !)
train_params = {
    'batch_size': 32,
    'grad_accum_steps': 4, # Simule batch 64
    'learning_rate': 6e-4,  # Vitesse standard
    'min_lr' : 3e-5, # Généralement 10% du LR max
    'warmup_iters' : 2000,
    'lr_decay_iters' : 100000, # Doit correspondre à la fin prévue de votre entraînement
    'eval_interval' : 10,
    'save_interval' : 100
}
# if __name__ == '__main__':

token = tk.BPETokenizer()
token.load_merges('data/ep_merges_full.json')
print(len(token.vocab))

trainer = gpt.ContinuousTrainer(
    model_class=gpt.GPTLanguageModel, # Class du model
    tokenizer=token,              # tokenizer
    config=model_config,
    train_params=train_params,
    data_root="data/train/wiki_fr_test",
    log_file="model/my_wiky_log.txt",
    ckpt_path="model/my_wiki.pth",
    history_path='model/my_wiky_history.json'
)

# Lance l'entraînement sur 20 fichiers puis s'arrête
#trainer.run_training_loop(max_files_session=8)
#trainer.run_training_loop_merge(50,1)
# CHANGEMENT de STratégie ...
trainer.train_bin()

In [ ]:
prompt = "Le wikipédia est "

In [ ]:
model=gpt.GenerateGPT(tokinizer=token,ckpt_path="model/my_wiki.pth")
model.load_for_inference()
out_ = model.generate_text(prompt=prompt, max_new_tokens=100,temperature=0.8)
print("-" * 40)
print(f"🤖 IA : {out_}")
print("-" * 40)

In [ ]:
visualizer = gpt.TrainingVisualizer('history.json') #('model/my_wiky_history.json')
visualizer.plot_metrics(window_size=10) # Lissage sur 10 fichiers

# Test Performance MPS/CPU

In [ ]:
import torch
import platform
import time

print(f"OS : {platform.system()} {platform.release()}")
print(f"Version PyTorch : {torch.__version__}")
print(f"MPS disponible : {torch.backends.mps.is_available()}")
print(f"MPS construit : {torch.backends.mps.is_built()}")

# Test de vitesse CPU vs MPS
size = 4000
x = torch.randn(size, size)

# Test CPU
start = time.time()
y_cpu = x @ x
print(f"⏱️ Temps CPU : {time.time() - start:.4f}s")

# Test MPS
if torch.backends.mps.is_available():
    x_mps = x.to("mps")
    start = time.time()
    y_mps = x_mps @ x_mps
    # On force la synchro pour mesurer le vrai temps
    torch.mps.synchronize() 
    print(f"⏱️ Temps MPS : {time.time() - start:.4f}s")

if torch.backends.mps.is_available():
    x_mps = torch.randn(size, size, device="mps")
    for i in range(5):
        start = time.time()
        y_mps = x_mps @ x_mps
        torch.mps.synchronize()
        print(f"Passage {i+1} - ⏱️ Temps MPS : {time.time() - start:.4f}s")

In [ ]:
print(len(token.vocab))